# Augmented Dickey-Fuller Test

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller

from config.config import Config
from src.utils import set_seed

In [2]:
cfg = Config(Path("../config/config.yaml"))
SEED = cfg.runtime.seed
HORIZON = cfg.runtime.horizon
rng = set_seed(SEED)

2025-08-29 20:29:08,417 - INFO - src.utils - Global random seed set to 42


In [3]:
df_full = pd.read_csv(Path(cfg.data.processed_dir) / "features_full.csv")

In [4]:
def adf_test(series: pd.Series, name: str = "series", as_dict: bool = False):
    s = pd.to_numeric(series, errors="coerce").dropna()
    stat, pval, lags, nobs, crit, _ = adfuller(s, autolag="AIC", regression="c")

    if as_dict:
        return {
            "name": name,
            "adf_stat": stat,
            "p_value": pval,
            "lags_used": lags,
            "n_obs": nobs,
            "crit_values": crit,
        }

    out = (
        f"ADF Test on '{name}'\n"
        f"{'-' * 40}\n"
        f"Test Statistic : {stat:.4f}\n"
        f"p-value        : {pval:.4g}\n"
        f"Lags Used      : {lags}\n"
        f"Observations   : {nobs}\n"
        f"{'-' * 40}\n"
    )
    for k, v in crit.items():
        out += f"Critical Value {k} : {v:.4f}\n"
    return out

In [5]:
adj = pd.to_numeric(df_full["adj_close"], errors="coerce")

print(adf_test(adj, "adj_close"))  # pretty print
print(adf_test(np.log(adj / adj.shift(1)), "log_return"))  # returns

ADF Test on 'adj_close'
----------------------------------------
Test Statistic : -0.5688
p-value        : 0.8779
Lags Used      : 20
Observations   : 1931
----------------------------------------
Critical Value 1% : -3.4337
Critical Value 5% : -2.8630
Critical Value 10% : -2.5676

ADF Test on 'log_return'
----------------------------------------
Test Statistic : -9.6846
p-value        : 1.18e-16
Lags Used      : 21
Observations   : 1929
----------------------------------------
Critical Value 1% : -3.4337
Critical Value 5% : -2.8630
Critical Value 10% : -2.5676

